# BDC Full Pipeline v4 — Mask Findings & R11 Per-Building Table Condition
Dataset: **DisasterM3_Bench Mirror V2** (masks + test_images + benchmark_release.json) + **R11 buildings.sqlite**.

**v4 additions:**
1. Condition `r11_image`: injects building counts and normalized bounding boxes directly from R11's champion SQLite table (`buildings.sqlite`, 8-view dihedral TTA, 20 px blob threshold).
   - Damaged questions use R11's damaged buildings; destroyed questions use R11's destroyed buildings.
   - Bounding boxes are scaled to 0–1000 using image dimensions; up to 10 listed in prompt.
   - Tested on fine-tuned 7B (subset M) and base 7B (subset MB).
   - Compared directly against H2 SQL counting agent on the same questions.
2. **Dual Answer Scoring:** Evaluates strict option letters (A–E) and mapped sentence numbers side-by-side with Wilson 95% confidence intervals.

| Name | Image | Findings Source | Purpose |
|---|---|---|---|
| `image_only` | yes | none | true baseline as deployed |
| `r11_image` | yes | R11 table (buildings.sqlite) | **deployable champion hybrid** |
| `unet_image` | yes | B1cls predicted U-Net mask | U-Net baseline hybrid |
| `oracle_image` | yes | ground-truth mask | upper bound with image |
| `oracle_text` | no | ground-truth mask | upper bound, text only |
| `text_only` | no | none | question and options only |


In [1]:
# ── CELL 1: Install dependencies (run this, then RESTART KERNEL) ───────────────
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'huggingface_hub>=0.28.0',
                'transformers>=4.49.0',
                'bitsandbytes>=0.46.1',
                'qwen-vl-utils',
                'segmentation-models-pytorch==0.3.4'], check=True)   # pinned: checkpoint key layout

print('✓ Done — NOW RESTART KERNEL before running Cell 2')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


✓ Done — NOW RESTART KERNEL before running Cell 2


In [2]:
# ── CELL 2: Config → locate dataset → funnel → oracle & R11 table counts ───
import json, glob, os, re, csv, gc, math, time, collections, sqlite3
import cv2
import numpy as np

# ══════════════════════════════ CONFIG ══════════════════════════════
SMOKE_TEST       = False # True = fast 40-entry check. Set False for the full run!
SMOKE_N          = 40
RUN_BASE_MODEL   = True    # also evaluate base Qwen2.5-VL-7B on sample MB
MIN_AREA_PX      = 30      # mask contour area threshold
MAX_BOXES        = 10      # boxes listed in the prompt (count uses all)
MAX_PIXELS       = 640 * 28 * 28
FT_MODEL         = 'AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP'
BASE_MODEL       = 'Qwen/Qwen2.5-VL-7B-Instruct'
OUT_DIR          = '/kaggle/working'
TAG              = '_smoke' if SMOKE_TEST else ''
T_START          = time.time()
TIME_LIMIT_HOURS = 11.0
BASE_SAMPLE      = 500
RESUME_DIR       = ''      # attach earlier v4 run output here if continuing

if RESUME_DIR and os.path.isdir(RESUME_DIR):
    import shutil
    for _p in glob.glob(os.path.join(RESUME_DIR, '**', 'bdc_*.csv'), recursive=True):
        _dst = os.path.join(OUT_DIR, os.path.basename(_p))
        if not os.path.exists(_dst):
            shutil.copy(_p, _dst); print('↺ restored', os.path.basename(_p))

if SMOKE_TEST:
    print('⚠ SMOKE_TEST = True → running fast 40-entry test per condition.\n')

# ══════════════════════ 1. Locate Datasets ══════════════════════
cands = sorted(set(glob.glob('/kaggle/input/**/benchmark_release.json', recursive=True)))
good = [c for c in cands if os.path.isdir(os.path.join(os.path.dirname(c), 'test_images'))]
if not good:
    raise FileNotFoundError('No benchmark_release.json with test_images/ found under /kaggle/input.')
BENCH_JSON = good[0]
ROOT       = os.path.dirname(BENCH_JSON)
IMG_DIR    = os.path.join(ROOT, 'test_images')

mask_dirs = glob.glob(os.path.join(ROOT, 'masks', 'test_building_damaged_mask')) or glob.glob(os.path.join(ROOT, '**', 'test_building_damaged_mask'), recursive=True)
MASK_DIR = mask_dirs[0]
DESTROYED_MASK_DIR = os.path.join(ROOT, 'masks', 'test_building_destroyed_mask')

# Locate R11 SQLite table
db_cands = sorted(set(glob.glob('/kaggle/input/**/buildings.sqlite', recursive=True)))
if not db_cands:
    if os.path.exists('Essential_task_delegation/Champion_Task/H2-results/H2/buildings.sqlite'):
        db_cands = ['Essential_task_delegation/Champion_Task/H2-results/H2/buildings.sqlite']
    else:
        raise FileNotFoundError('buildings.sqlite not found. Attach H2 results or buildings dataset under /kaggle/input.')
DB_PATH = db_cands[0]

h2_cands = sorted(set(glob.glob('/kaggle/input/**/agent_eval.csv', recursive=True)))
H2_EVAL_PATH = h2_cands[0] if h2_cands else (
    'Essential_task_delegation/Champion_Task/H2-results/H2/agent_eval.csv'
    if os.path.exists('Essential_task_delegation/Champion_Task/H2-results/H2/agent_eval.csv') else None
)

print(f'✓ ROOT     = {ROOT}\n✓ MASK_DIR (damaged)   = {MASK_DIR}\n✓ MASK_DIR (destroyed) = {DESTROYED_MASK_DIR}')
print(f'✓ IMG_DIR  = {IMG_DIR}\n✓ DB_PATH  = {DB_PATH}')
print(f'✓ H2_EVAL  = {H2_EVAL_PATH}')

# ══════════════════════ 2. Index Images and Masks ══════════════════════
IMG_EXT = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

def stem(p):
    return os.path.splitext(str(p).replace('\\', '/').split('/')[-1])[0]

def index_dir(d):
    idx = {}
    for r, _, fs in os.walk(d):
        for f in fs:
            if os.path.splitext(f)[1].lower() in IMG_EXT:
                idx.setdefault(os.path.splitext(f)[0], os.path.join(r, f))
    return idx

img_idx   = index_dir(IMG_DIR)
mask_idx  = index_dir(MASK_DIR)
dmask_idx = index_dir(DESTROYED_MASK_DIR)

def resolve_image(p):
    q = str(p).replace('\\', '/')
    for base in (ROOT, IMG_DIR):
        c = os.path.join(base, q)
        if os.path.isfile(c): return c
    return img_idx.get(stem(q))

def find_mask(post_image_path, idx):
    s = stem(post_image_path)
    cands = [s.replace('_post_disaster', '').replace('_post', ''), s, s.replace('_post_disaster', '_post')]
    for c in cands:
        if c in idx: return idx[c]
    return None

def mask_to_count_boxes(mask):
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    h, w = mask.shape
    boxes = []
    for c in contours:
        if cv2.contourArea(c) < MIN_AREA_PX: continue
        x, y, bw, bh = cv2.boundingRect(c)
        boxes.append([int(y / h * 1000), int(x / w * 1000), int((y + bh) / h * 1000), int((x + bw) / w * 1000)])
    return len(boxes), boxes

# ══════════════════════ 3. Load R11 Table Data ══════════════════════
con = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
cur = con.cursor()
r11_images = {r[0]: (r[1], r[2]) for r in cur.execute('SELECT image, width, height FROM images').fetchall()}
r11_buildings = collections.defaultdict(list)
for r in cur.execute('SELECT image, class_name, bbox_x, bbox_y, bbox_w, bbox_h FROM buildings'):
    im, cname, bx, by, bw, bh = r
    r11_buildings[(im, cname)].append((bx, by, bw, bh))
con.close()
print(f'✓ R11 DB loaded: {len(r11_images):,} images, {sum(len(v) for v in r11_buildings.values()):,} buildings')

# ══════════════════════ 4. Benchmark Funnel & Annotation ══════════════════════
with open(BENCH_JSON, encoding='utf-8') as f:
    bench = json.load(f)

# ── NEW ──
bdc_all = [e for e in bench if e.get('task') == 'Building Damage Counting']
for i, e in enumerate(bdc_all):
    e['id'] = f"building_damage_counting_{i}"

KEEP = ['Damaged building counting.', 'Totally-destroyed building counting.']
CLS_DESTROYED = 'Totally-destroyed building counting.'
ALL = [e for e in bdc_all if e.get('post_image_type') == 'Optical' and e.get('cls_description') in KEEP]

for i, e in enumerate(ALL):
    e['uid'] = i
    post_name = str(e['post_image_path']).replace('\\', '/').split('/')[-1]
    e['image_filename'] = post_name
    e['image_path'] = resolve_image(e['post_image_path'])
    
    # 1. Oracle mask findings
    mp = find_mask(e['post_image_path'], dmask_idx if e['cls_description'] == CLS_DESTROYED else mask_idx)
    mk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE) if mp else None
    if mk is not None:
        cnt, bxs = mask_to_count_boxes(mk)
        e.update({'mask_path': mp, 'oracle_count': cnt, 'oracle_boxes': bxs})
    else:
        e.update({'mask_path': None, 'oracle_count': -1, 'oracle_boxes': []})
        
    # 2. R11 Table findings
    target_class = 'destroyed' if e['cls_description'] == CLS_DESTROYED else 'damaged'
    if post_name in r11_images:
        w, h = r11_images[post_name]
        blobs = r11_buildings.get((post_name, target_class), [])
        r11_bxs = []
        for bx, by, bw, bh in blobs:
            r11_bxs.append([
                max(0, min(1000, int(by / h * 1000))),
                max(0, min(1000, int(bx / w * 1000))),
                max(0, min(1000, int((by + bh) / h * 1000))),
                max(0, min(1000, int((bx + bw) / w * 1000)))
            ])
        e.update({'in_r11_table': True, 'r11_count': len(blobs), 'r11_boxes': r11_bxs})
    else:
        e.update({'in_r11_table': False, 'r11_count': -1, 'r11_boxes': []})

WITH_IMG = [e for e in ALL if e['image_path']]
M        = [e for e in ALL if e['image_path'] and e['oracle_count'] >= 0]
R11_SUB  = [e for e in ALL if e['image_path'] and e['in_r11_table']]

print(f'\n── DATA FUNNEL ──')
print(f'Total Optical Damaged/Destroyed : {len(ALL)}')
print(f'With Test Image                 : {len(WITH_IMG)}')
print(f'Subset M (Image + Oracle Mask)  : {len(M)}')
print(f'In R11 Table                    : {len(R11_SUB)} (Missing: {len(ALL) - len(R11_SUB)})')

MB = M if len(M) <= BASE_SAMPLE else [M[i] for i in np.unique(np.linspace(0, len(M) - 1, BASE_SAMPLE).astype(int))]
print(f'Base-Model Sample MB            : {len(MB)}')

with open(f'{OUT_DIR}/augmented_bdc_benchmark_v4.json', 'w') as f:
    json.dump(ALL, f, indent=1)
print(f'✓ Benchmark manifest saved → {OUT_DIR}/augmented_bdc_benchmark_v4.json')


✓ ROOT     = /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2
✓ MASK_DIR (damaged)   = /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2/masks/test_building_damaged_mask
✓ MASK_DIR (destroyed) = /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2/masks/test_building_destroyed_mask
✓ IMG_DIR  = /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2/test_images
✓ DB_PATH  = /kaggle/input/datasets/aryansami6/agent-eval/buildings.sqlite
✓ H2_EVAL  = /kaggle/input/datasets/aryansami6/agent-eval/agent_eval.csv
✓ R11 DB loaded: 2,652 images, 117,687 buildings

── DATA FUNNEL ──
Total Optical Damaged/Destroyed : 1703
With Test Image                 : 1703
Subset M (Image + Oracle Mask)  : 1701
In R11 Table                    : 1702 (Missing: 1)
Base-Model Sample MB            : 500
✓ Benchmark manifest saved → /kaggle/working/augmented_bdc_benchmark_v4.json


In [3]:
# ── CELL 2b (diagnostic, no GPU): are findings right for BOTH question types? ──
def to_int(x):
    m = re.search(r'-?\d+', str(x))
    return int(m.group()) if m else None

mdir = os.path.join(ROOT, 'masks')
if os.path.isdir(mdir):
    print('masks/ contains:')
    for d in sorted(os.listdir(mdir)):
        p = os.path.join(mdir, d)
        print(f'   {d:45s} {len(os.listdir(p)) if os.path.isdir(p) else "file"}')

# 1. Oracle mask audit
by_oracle = collections.defaultdict(list)
for e in M:
    g = to_int(e['ground_truth'])
    if g is not None: by_oracle[e['cls_description']].append((e['oracle_count'], g))

print('\n── 1. Oracle count vs benchmark answer, per question type: ──')
for cls, pr in by_oracle.items():
    exact = np.mean([a == b for a, b in pr]) * 100
    over  = np.mean([a > b for a, b in pr]) * 100
    under = np.mean([a < b for a, b in pr]) * 100
    print(f'   {cls:40s} n={len(pr):4d} | exact {exact:5.1f}% | mask>GT {over:5.1f}% | mask<GT {under:5.1f}% | '
          f'mean signed error {np.mean([a - b for a, b in pr]):+.2f}')

# 2. R11 SQLite table audit
by_r11 = collections.defaultdict(list)
for e in R11_SUB:
    g = to_int(e['ground_truth'])
    if g is not None and e['r11_count'] >= 0:
        by_r11[e['cls_description']].append((e['r11_count'], g))

print('\n── 2. R11 table count vs benchmark answer, per question type: ──')
for cls, pr in by_r11.items():
    exact = np.mean([a == b for a, b in pr]) * 100
    over  = np.mean([a > b for a, b in pr]) * 100
    under = np.mean([a < b for a, b in pr]) * 100
    mae   = np.mean([abs(a - b) for a, b in pr])
    print(f'   {cls:40s} n={len(pr):4d} | exact {exact:5.1f}% | R11>GT {over:5.1f}% | R11<GT {under:5.1f}% | '
          f'MAE {mae:.2f} | mean signed error {np.mean([a - b for a, b in pr]):+.2f}')

masks/ contains:
   flooding_mask                                 117
   intact_buildings_near_flooding                117
   intact_buildings_near_lava                    37
   test_building_damaged_mask                    2855
   test_building_destroyed_mask                  2855
   test_building_intact_mask                     2855
   test_road_debris_covered_mask                 2072
   test_road_flooded_mask                        2072
   test_road_intact_mask                         2072
   volcano_lava                                  39

── 1. Oracle count vs benchmark answer, per question type: ──
   Damaged building counting.               n= 972 | exact  94.5% | mask>GT   0.0% | mask<GT   5.5% | mean signed error -0.10
   Totally-destroyed building counting.     n= 729 | exact  93.0% | mask>GT   0.0% | mask<GT   7.0% | mean signed error -0.13

── 2. R11 table count vs benchmark answer, per question type: ──
   Damaged building counting.               n= 973 | exact   9.0% | 

In [4]:
# ── CELL 3: Model loader + prompt builder + inference runner ───────────────────
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

def load_vlm(model_id):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    proc = AutoProcessor.from_pretrained(model_id, trust_remote_code=True,
                                         min_pixels=64 * 28 * 28, max_pixels=MAX_PIXELS)
    vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    vlm.eval()
    print(f'✓ {model_id} loaded | Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')
    return vlm, proc

def build_text(entry, count=None, boxes=None):
    q = str(entry['prompts']).replace('<image>', '').strip()
    question = (f'--- Question ---\n{q}\n{entry["options_str"]}\n\n'
                f"Output only the single best letter (e.g., 'The answer is B.').")
    if count is None or count < 0:
        return question
    kind = entry['cls_description'].lower().replace('.', '').replace(' counting', '')
    if boxes:
        box_str = '\n'.join(f'  - Building {i + 1}: [ymin={b[0]}, xmin={b[1]}, ymax={b[2]}, xmax={b[3]}]'
                            for i, b in enumerate(boxes[:MAX_BOXES]))
        if count > MAX_BOXES:
            box_str += f'\n  (first {MAX_BOXES} of {count} listed)'
    else:
        box_str = '  - None detected.'
    return (f'[Segmentation Analysis]:\n- {kind} count: {count}\n'
            f'- Bounding boxes (0-1000 scale):\n{box_str}\n\n{question}')

def extract_strict_letter(resp):
    m = re.search(r'[Aa]nswer\s*(?:is|:)?\s*\(?([A-E])\b', resp)
    if m: return m.group(1)
    m = re.search(r'\b([A-E])\b', resp)
    return m.group(1) if m else '?'

NUM_WORDS = {'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4,
             'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
             'no': 0, 'none': 0}

def map_sentence_number(resp, options_list):
    r = resp.lower()
    val = None
    for w, num in NUM_WORDS.items():
        if re.search(rf'\b{w}\b', r):
            val = num; break
    if val is None:
        nums = re.findall(r'\b\d+\b', r)
        if nums: val = int(nums[-1])
    if val is not None and options_list and val in options_list:
        return 'ABCDE'[options_list.index(val)], val
    return '?', val

@torch.no_grad()
def ask(vlm, proc, text, img=None):
    content = ([{'type': 'image'}] if img is not None else []) + [{'type': 'text', 'text': text}]
    prompt = proc.apply_chat_template([{'role': 'user', 'content': content}],
                                      tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[prompt], images=[img] if img is not None else None,
                  padding=True, return_tensors='pt').to(vlm.device)
    out = vlm.generate(**inputs, max_new_tokens=32, do_sample=False, temperature=None, top_p=None)
    return proc.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()

FIELDS = ['uid', 'id', 'image', 'cls', 'ground_truth', 'gt_option', 'pred_option', 'correct',
          'mapped_option', 'mapped_correct', 'count_used', 'response']

def run_experiment(vlm, proc, model_tag, name, entries, use_image, count_src):
    path = f'{OUT_DIR}/bdc_{model_tag}_{name}{TAG}.csv'
    done = set()
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            done = {int(r['uid']) for r in csv.DictReader(f)}
        print(f'↺ Resuming {name} ({len(done)} already finished)')
    
    f_out = open(path, 'a', newline='', encoding='utf-8')
    writer = csv.DictWriter(f_out, fieldnames=FIELDS)
    if not done:
        writer.writeheader()
    
    t0 = time.time()
    for i, e in enumerate(entries):
        if e['uid'] in done: continue
        
        cnt, bxs = None, None
        if count_src == 'oracle':
            cnt, bxs = e['oracle_count'], e['oracle_boxes']
        elif count_src == 'r11':
            cnt, bxs = e['r11_count'], e['r11_boxes']
        elif count_src == 'unet':
            cnt, bxs = e.get('unet_count'), e.get('unet_boxes')
            
        txt = build_text(e, count=cnt, boxes=bxs)
        img = Image.open(e['image_path']).convert('RGB') if use_image and e['image_path'] else None
        
        resp = ask(vlm, proc, txt, img=img)
        strict_opt = extract_strict_letter(resp)
        strict_hit = (strict_opt == e['ground_truth_option'])
        
        mapped_opt, _ = map_sentence_number(resp, e.get('options_list', []))
        final_mapped = strict_opt if strict_opt != '?' else mapped_opt
        mapped_hit = (final_mapped == e['ground_truth_option'])
        
        row = {
            'uid': e['uid'],
            'id': e['id'],
            'image': e['post_image_path'],
            'cls': e['cls_description'],
            'ground_truth': e['ground_truth'],
            'gt_option': e['ground_truth_option'],
            'pred_option': strict_opt,
            'correct': str(strict_hit),
            'mapped_option': final_mapped,
            'mapped_correct': str(mapped_hit),
            'count_used': cnt if cnt is not None else '',
            'response': resp
        }
        writer.writerow(row)
        f_out.flush()
        
        if (i + 1) % 50 == 0 or (i + 1) == len(entries):
            print(f"  [{i+1}/{len(entries)}] {name} | strict: {strict_opt} | mapped: {final_mapped} | gt: {e['ground_truth_option']}")
            
    f_out.close()
    elapsed = (time.time() - t0) / 60
    print(f'✓ Finished {model_tag}_{name} in {elapsed:.1f} min')
    return True


In [5]:
# ── CELL 4: Real U-Net → predicted damage mask → count + boxes ─────────────────
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from huggingface_hub import hf_hub_download

# ═════════════ WHICH U-NET? (edit this block) ═════════════
# Currently reachable: the B1 baseline (ResNet-34, Resize-512). UNET_LABEL names the result file, so
# results from different checkpoints never overwrite or get mixed with each other.
UNET_LABEL  = 'B1cls'   # B1 checkpoint, per-question-type classes (separate result file from the earlier 'B1' run)
UNET_SOURCE = dict(
    hf_repo='AbrarAlam/disasterm3-unet-checkpoints-2', filename='best_model.pth',
    local_path=None,            # or '/kaggle/input/<your-dataset>/best_model.pth'
    input_mode='auto',          # D4-style checkpoints (they store cfg) are detected -> 'native'
)
# D4 run (R7 or whichever run is the champion) once you have its last.pth as a Kaggle dataset:
# UNET_LABEL  = 'R7'
# UNET_SOURCE = dict(hf_repo=None, filename=None, local_path='/kaggle/input/<dataset>/last.pth', input_mode='auto')
# (the load line should then say Unet+resnet50 | input_mode=native | D4-style checkpoint=True | epoch=39)

ENABLE_UNET = True    # False → skip the unet_image experiment
UNET_TTA    = False   # True = 8-view dihedral TTA (8x slower)
UNET_CLEAN  = True    # close+open 3x3 on the predicted mask (same cleaning as unet_findings_to_vlm.py)

def _hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')

_UNET = {'model': None, 'mode': None}
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
_ARCH = {'Unet': smp.Unet, 'UnetPlusPlus': smp.UnetPlusPlus,
         'DeepLabV3Plus': smp.DeepLabV3Plus, 'MAnet': smp.MAnet}

def detect_arch_encoder(state_keys):
    arch = 'UnetPlusPlus' if any('decoder.blocks.x_' in k for k in state_keys) else 'Unet'
    enc  = 'resnet50' if any(k.endswith('layer1.0.conv3.weight') for k in state_keys) else 'resnet34'
    return arch, enc

def _ensure_unet():
    if _UNET['model'] is not None: return
    path = UNET_SOURCE['local_path'] or hf_hub_download(
        UNET_SOURCE['hf_repo'], UNET_SOURCE['filename'], token=_hf_token())
    ck = torch.load(path, map_location='cpu', weights_only=False)
    is_d4 = isinstance(ck, dict) and 'cfg' in ck                      # D4 notebook checkpoints store cfg
    state = ck['model_state_dict'] if isinstance(ck, dict) and 'model_state_dict' in ck else ck
    state = {k.replace('module.', '', 1): v for k, v in state.items()}
    if is_d4:
        arch, enc = ck['cfg']['arch'], ck['cfg']['encoder']
    else:
        arch, enc = detect_arch_encoder(list(state.keys()))
    mode = UNET_SOURCE['input_mode']
    if mode == 'auto':
        mode = 'native' if is_d4 else 'resize512'
    m = _ARCH[arch](encoder_name=enc, encoder_weights=None, in_channels=3, classes=4)
    try:
        m.load_state_dict(state)
    except Exception as ex:
        raise RuntimeError(f'Could not load {path} as {arch}+{enc}: {str(ex)[:400]}\n'
                           'Set arch/encoder manually (e.g. DeepLabV3Plus or MAnet) in _ensure_unet().')
    _UNET['model'] = m.cuda().eval()
    _UNET['mode']  = mode
    ep = ck.get('epoch') if isinstance(ck, dict) else None
    print(f'✓ U-Net loaded: {arch}+{enc} | input_mode={mode} | D4-style checkpoint={is_d4} | epoch={ep}\n'
          f'  source: {path}')

def _fwd(x):
    h, w = x.shape[-2:]
    net = _UNET['model']
    if _UNET['mode'] == 'resize512':
        lg = net(F.interpolate(x, size=(512, 512), mode='bilinear', align_corners=False))
        return F.interpolate(lg.float(), size=(h, w), mode='bilinear', align_corners=False)
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw: x = F.pad(x, (0, pw, 0, ph), mode='reflect')
    return net(x)[..., :h, :w].float()

_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]
def _tf(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x
def _inv(y, k, flip):
    if flip: y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def unet_predict_classmap(image_path):
    _ensure_unet()
    bgr = cv2.imread(image_path)
    if bgr is None: raise FileNotFoundError(image_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).unsqueeze(0)
    x = ((x - _MEAN) / _STD).cuda()
    with torch.autocast('cuda', dtype=torch.float16):
        if UNET_TTA:
            prob = sum(torch.softmax(_inv(_fwd(_tf(x, k, fl)), k, fl), dim=1) for k, fl in _VIEWS)
            cls = prob.argmax(1)
        else:
            cls = _fwd(x).argmax(1)
    return cls[0].cpu().numpy().astype(np.uint8)       # 0=BG 1=Intact 2=Damaged 3=Destroyed

_CLS_CACHE = {}
def _classmap_cached(path):
    if path not in _CLS_CACHE:
        if len(_CLS_CACHE) > 200: _CLS_CACHE.clear()
        _CLS_CACHE[path] = unet_predict_classmap(path)
    return _CLS_CACHE[path]

def _unet_mask_for(entry):
    # damaged question -> predicted class 2 ; destroyed question -> predicted class 3 (matches the GT masks)
    target = 3 if entry['cls_description'] == CLS_DESTROYED else 2
    m = ((_classmap_cached(entry['image_path']) == target).astype(np.uint8)) * 255
    if UNET_CLEAN:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        m = cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_CLOSE, k), cv2.MORPH_OPEN, k)
    return m

UNET_PREDICT_MASK = _unet_mask_for if ENABLE_UNET else None
_unet_cache = {}

def get_unet_counts(entry):
    key = entry['uid']
    if key not in _unet_cache:
        _unet_cache[key] = mask_to_count_boxes(_unet_mask_for(entry))
    return _unet_cache[key]

def unet_sanity_check(n=20):
    # Compares the predicted damage mask with the ground-truth mask on a few entries of subset M.
    sub_ = M[::max(1, len(M) // n)][:n]
    ious, diffs = [], []
    for e in sub_:
        pm = _unet_mask_for(e)
        gm = cv2.imread(e['mask_path'], cv2.IMREAD_GRAYSCALE)
        if gm.shape != pm.shape:
            gm = cv2.resize(gm, (pm.shape[1], pm.shape[0]), interpolation=cv2.INTER_NEAREST)
        gb, pb = gm > 127, pm > 127
        union = (gb | pb).sum()
        ious.append((gb & pb).sum() / union if union else 1.0)
        diffs.append(mask_to_count_boxes(pm)[0] - e['oracle_count'])
    print(f'── U-Net sanity check on {len(sub_)} entries of M ──')
    print(f'   mean pixel IoU (predicted class vs GT mask of the question type): {np.mean(ious):.3f}')
    print(f'   predicted count − oracle count: mean {np.mean(diffs):+.1f} | mean abs {np.mean(np.abs(diffs)):.1f}')
    print('   (low IoU / large count gap → the unet_image result will be far below oracle_image)')

if ENABLE_UNET:
    try:
        unet_sanity_check(20)
    except Exception as ex:
        print(f'⚠ U-Net check failed: {ex}')
else:
    print('U-Net hook disabled (ENABLE_UNET = False)')

best_model.pth: reconstructing file:   0%|          |  0.00B / 97.9MB            

best_model.pth: downloading bytes:           |  0.00B            

✓ U-Net loaded: Unet+resnet34 | input_mode=resize512 | D4-style checkpoint=False | epoch=None
  source: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-unet-checkpoints-2/snapshots/2a34c58a145ee5f996aa9376339f437e5dec5781/best_model.pth
── U-Net sanity check on 20 entries of M ──
   mean pixel IoU (predicted class vs GT mask of the question type): 0.133
   predicted count − oracle count: mean -13.3 | mean abs 19.1
   (low IoU / large count gap → the unet_image result will be far below oracle_image)


In [6]:
# ── CELL 4b (diagnostic, run once): why is the U-Net IoU low? ──────────────────
import matplotlib.pyplot as plt

def read_mask(path):
    if not path or not os.path.exists(path): return None
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if m is not None: return m
    try:
        from PIL import Image
        return np.array(Image.open(path).convert('L'))
    except Exception:
        return None

def _iou(a, b):
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else float('nan')

def unet_diagnose(n=16, show=True):
    sub_ = M[:n]
    ious_c2, ious_c3 = [], []
    pred_area_c2, gt_area_c2 = [], []
    pred_area_c3, gt_area_c3 = [], []
    shape_mismatch = 0
    
    for e in sub_:
        cls = _classmap_cached(e['image_path'])
        gm = read_mask(e['mask_path'])
        if gm is None: continue
        if gm.shape != cls.shape:
            shape_mismatch += 1
            gm = cv2.resize(gm, (cls.shape[1], cls.shape[0]), interpolation=cv2.INTER_NEAREST)
        
        # Ground truth binary mask (>127)
        gt_bin = (gm > 127)
        tot_pix = gm.shape[0] * gm.shape[1]
        
        if e['cls_description'] == CLS_DESTROYED:
            pred_bin = (cls == 3)
            ious_c3.append(_iou(pred_bin, gt_bin))
            pred_area_c3.append(pred_bin.sum() / tot_pix * 100)
            gt_area_c3.append(gt_bin.sum() / tot_pix * 100)
        else:
            pred_bin = (cls == 2)
            ious_c2.append(_iou(pred_bin, gt_bin))
            pred_area_c2.append(pred_bin.sum() / tot_pix * 100)
            gt_area_c2.append(gt_bin.sum() / tot_pix * 100)
            
    print(f"Diagnostics on {len(sub_)} samples (shape mismatches resized: {shape_mismatch}):")
    if ious_c2:
        print(f"  Damaged   (class 2): mean IoU = {np.nanmean(ious_c2):.4f} | "
              f"pred area = {np.mean(pred_area_c2):.2f}% vs GT area = {np.mean(gt_area_c2):.2f}%")
    if ious_c3:
        print(f"  Destroyed (class 3): mean IoU = {np.nanmean(ious_c3):.4f} | "
              f"pred area = {np.mean(pred_area_c3):.2f}% vs GT area = {np.mean(gt_area_c3):.2f}%")

unet_diagnose()

Diagnostics on 16 samples (shape mismatches resized: 0):
  Damaged   (class 2): mean IoU = 0.0512 | pred area = 1.83% vs GT area = 7.53%


In [7]:
# ── CELL 4c: when continuing an earlier run (RESUME_DIR set), redo ONLY the destroyed questions ──
# The earlier oracle runs gave destroyed questions the DAMAGED-mask count. Damaged rows are still valid and are kept.
if RESUME_DIR:
    for nm in ('ft_oracle_text', 'ft_oracle_image', 'base_oracle_image'):
        p = f'{OUT_DIR}/bdc_{nm}{TAG}.csv'
        if not os.path.exists(p): continue
        rows = list(csv.DictReader(open(p)))
        keep = [r for r in rows if r['cls'] != CLS_DESTROYED]
        with open(p, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=FIELDS); w.writeheader(); w.writerows(keep)
        print(f'{nm}: kept {len(keep)} damaged rows; {len(rows) - len(keep)} destroyed rows will be re-run with the destroyed mask')
else:
    print('RESUME_DIR not set → fresh run, nothing to trim.')

RESUME_DIR not set → fresh run, nothing to trim.


In [8]:
# ── CELL 5: Run all experiments (resumable, including r11_image) ───────────────
def sub(lst):
    if not SMOKE_TEST: return lst
    return lst[::max(1, len(lst) // SMOKE_N)][:SMOKE_N]

SUBSETS = {'M': sub(M), 'IMG': sub(WITH_IMG), 'MB': sub(MB), 'R11': sub(R11_SUB)}

EXPERIMENTS = [
    ('ft',   'image_only',                 'IMG', True,  None),       # true baseline
    ('ft',   'r11_image',                  'M',   True,  'r11'),      # <-- R11 table hybrid
    ('ft',   'oracle_image',               'M',   True,  'oracle'),   # upper bound with image
    ('ft',   f'unet-{UNET_LABEL}_image',   'IMG', True,  'unet'),     # B1cls U-Net hybrid
    ('ft',   'text_only',                  'M',   False, None),
    ('ft',   'oracle_text',                'M',   False, 'oracle'),
    ('base', 'image_only',                 'MB',  True,  None),
    ('base', 'r11_image',                  'MB',  True,  'r11'),      # <-- Base on R11 table
    ('base', 'oracle_image',               'MB',  True,  'oracle'),
]
if not ENABLE_UNET:
    EXPERIMENTS = [x for x in EXPERIMENTS if x[4] != 'unet']
if not RUN_BASE_MODEL:
    EXPERIMENTS = [x for x in EXPERIMENTS if x[0] != 'base']

stopped = False
for tag, model_id in (('ft', FT_MODEL), ('base', BASE_MODEL)):
    exps = [x for x in EXPERIMENTS if x[0] == tag]
    if not exps or stopped: continue
    vlm, proc = load_vlm(model_id)
    for (_, name, subset, use_img, csrc) in exps:
        if not run_experiment(vlm, proc, tag, name, SUBSETS[subset], use_img, csrc):
            stopped = True
            break
    del vlm, proc
    gc.collect(); torch.cuda.empty_cache()

print('\n⏱ Stopped early on time budget' if stopped else '\n✓ All experiments finished successfully.')


preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

✓ Qwen/Qwen2.5-VL-7B-Instruct loaded | Free VRAM: 12.5 GB


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


  [50/500] image_only | strict: B | mapped: B | gt: C
  [100/500] image_only | strict: D | mapped: D | gt: D
  [150/500] image_only | strict: E | mapped: E | gt: E
  [200/500] image_only | strict: ? | mapped: ? | gt: A
  [250/500] image_only | strict: E | mapped: E | gt: B
  [300/500] image_only | strict: ? | mapped: ? | gt: A
  [350/500] image_only | strict: B | mapped: B | gt: A
  [400/500] image_only | strict: A | mapped: A | gt: C
  [450/500] image_only | strict: E | mapped: E | gt: A
  [500/500] image_only | strict: B | mapped: B | gt: A
✓ Finished base_image_only in 17.4 min
  [50/500] r11_image | strict: A | mapped: A | gt: C
  [100/500] r11_image | strict: B | mapped: B | gt: D
  [150/500] r11_image | strict: E | mapped: E | gt: E
  [200/500] r11_image | strict: A | mapped: A | gt: A
  [250/500] r11_image | strict: A | mapped: A | gt: B
  [300/500] r11_image | strict: A | mapped: A | gt: A
  [350/500] r11_image | strict: E | mapped: E | gt: A
  [400/500] r11_image | strict: A |

In [9]:
# ── CELL 6: Summary table (Strict vs Mapped Accuracy & H2 Comparison) ───────────
def wilson(k, n, z=1.96):
    if n == 0: return 0.0, 0.0
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - a) / d, (c + a) / d

CLS_D, CLS_T = 'Damaged building counting.', 'Totally-destroyed building counting.'
M_UIDS  = {e['uid'] for e in M}
MB_UIDS = {e['uid'] for e in MB}

# Load H2 agent evaluation lookup if present
h2_lookup = {}
if H2_EVAL_PATH and os.path.exists(H2_EVAL_PATH):
    with open(H2_EVAL_PATH, encoding='utf-8') as f:
        for r in csv.DictReader(f):
            if r.get('llm') == FT_MODEL or 'disasterm3' in r.get('llm', ''):
                h2_lookup[r['id']] = (r['h2_correct'] == 'True')
    print(f'✓ Loaded H2 agent evaluations for {len(h2_lookup)} questions.')

files = sorted(glob.glob(f'{OUT_DIR}/bdc_*{TAG}.csv'))
files = [p for p in files if 'summary' not in p and (TAG or not p.endswith('_smoke.csv'))]

table = []
for p in files:
    base = os.path.basename(p)[4:-4]
    if TAG: base = base[:-len(TAG)]
    model, name = base.split('_', 1)
    if model not in ('ft', 'base'): continue
    with open(p, encoding='utf-8') as f: rows = list(csv.DictReader(f))
    n = len(rows)
    strict_k = sum(r['correct'] == 'True' for r in rows)
    mapped_k = sum(r.get('mapped_correct', r['correct']) == 'True' for r in rows)
    unparsed = sum(r['pred_option'] == '?' for r in rows)
    num_mapped = sum(r['pred_option'] == '?' and r.get('mapped_option', '?') != '?' for r in rows)
    
    s_acc = strict_k / n * 100 if n else 0
    m_acc = mapped_k / n * 100 if n else 0
    s_lo, s_hi = wilson(strict_k, n)
    m_lo, m_hi = wilson(mapped_k, n)
    
    # On subset M
    m_rows = [r for r in rows if int(r['uid']) in M_UIDS]
    s_km = sum(r['correct'] == 'True' for r in m_rows)
    m_km = sum(r.get('mapped_correct', r['correct']) == 'True' for r in m_rows)
    s_am = s_km / len(m_rows) * 100 if m_rows else 0
    m_am = m_km / len(m_rows) * 100 if m_rows else 0
    
    # On subset MB
    mb_rows = [r for r in rows if int(r['uid']) in MB_UIDS]
    s_kmb = sum(r['correct'] == 'True' for r in mb_rows)
    m_kmb = sum(r.get('mapped_correct', r['correct']) == 'True' for r in mb_rows)
    s_amb = s_kmb / len(mb_rows) * 100 if mb_rows else 0
    m_amb = m_kmb / len(mb_rows) * 100 if mb_rows else 0
    
    # H2 accuracy on identical questions
    h2_matched = [h2_lookup[r['id']] for r in rows if r.get('id') in h2_lookup]
    h2_acc_str = f"{sum(h2_matched)/len(h2_matched)*100:.1f}% (n={len(h2_matched)})" if h2_matched else "—"
    
    table.append({
        'model': model, 'experiment': name, 'n': n,
        'strict_acc_%': round(s_acc, 2), 'strict_ci95': f'{s_lo * 100:.1f}-{s_hi * 100:.1f}',
        'mapped_acc_%': round(m_acc, 2), 'mapped_ci95': f'{m_lo * 100:.1f}-{m_hi * 100:.1f}',
        'n_on_M': len(m_rows), 'strict_acc_M_%': round(s_am, 2), 'mapped_acc_M_%': round(m_am, 2),
        'n_on_MB': len(mb_rows), 'strict_acc_MB_%': round(s_amb, 2), 'mapped_acc_MB_%': round(m_amb, 2),
        'unparsed': unparsed, 'num_mapped': num_mapped, 'h2_same_q': h2_acc_str
    })

order = {'text_only': 0, 'image_only': 1, 'r11_image': 2, 'unet-B1cls_image': 3, 'oracle_text': 4, 'oracle_image': 5}
table.sort(key=lambda r: (r['model'], order.get(r['experiment'], 9), r['experiment']))

cols = list(table[0].keys()) if table else []
print('\n' + '| ' + ' | '.join(cols) + ' |')
print('|' + '---|' * len(cols))
for r in table:
    print('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')

with open(f'{OUT_DIR}/bdc_summary_v4{TAG}.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(table)
print(f'\n✓ Saved summary table → {OUT_DIR}/bdc_summary_v4{TAG}.csv')


✓ Loaded H2 agent evaluations for 100 questions.

| model | experiment | n | strict_acc_% | strict_ci95 | mapped_acc_% | mapped_ci95 | n_on_M | strict_acc_M_% | mapped_acc_M_% | n_on_MB | strict_acc_MB_% | mapped_acc_MB_% | unparsed | num_mapped | h2_same_q |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| base | image_only | 500 | 35.2 | 31.1-39.5 | 35.2 | 31.1-39.5 | 500 | 35.2 | 35.2 | 500 | 35.2 | 35.2 | 17 | 0 | 45.5% (n=22) |
| base | r11_image | 500 | 30.2 | 26.3-34.4 | 30.2 | 26.3-34.4 | 500 | 30.2 | 30.2 | 500 | 30.2 | 30.2 | 0 | 0 | 45.5% (n=22) |
| base | oracle_image | 500 | 89.2 | 86.2-91.6 | 89.2 | 86.2-91.6 | 500 | 89.2 | 89.2 | 500 | 89.2 | 89.2 | 0 | 0 | 45.5% (n=22) |
| ft | text_only | 1701 | 21.11 | 19.2-23.1 | 21.11 | 19.2-23.1 | 1701 | 21.11 | 21.11 | 500 | 21.0 | 21.0 | 0 | 0 | 39.4% (n=66) |
| ft | image_only | 1703 | 15.56 | 13.9-17.4 | 16.85 | 15.1-18.7 | 1701 | 15.58 | 16.81 | 500 | 18.0 | 18.6 | 192 | 192 | 39.4% (n=66) |
| ft | r11_image